# Repeated-structure partial ordered registration

**Goal check:** reconstruct a canonical ordered PT structure from unordered cuts
of repeated physical realizations. Pooled paths can move with coverage; instead
fit each calibration point cloud separately and infer ordered molecular
correspondence to a source reference. Unknown nephron memberships contribute to
local variation, not separately reconstructed individual nephrons.

This tests a mean-path special case of the shared distributional structure.
Registration is existing methodology, not a new-mathematics claim. A partial
query need not cover the full reference and specimen occupancy is never matched.


## Protocol and evidence boundary

The protocol was written in the research ledger before private-data fitting.
Source-only count PCs, healthy controls, gene folds, marker exclusions and
calibration/evaluation halves remain fixed. Query calibration labels only select
the S3-only stress population; neither curve nor registration receives them.

Fit a generic unlabeled query curve, average its local expression at 31 native
knots (bandwidth .08; effective support >=12), and match those means to 101 source
mean knots with nondecreasing indices and free reference endpoints. Compare both
query orientations using modeling expression alone. Reference indices may repeat
or jump. Every query knot is forced to match; collapses and costs are diagnostics,
not confident anatomy. This is a healthy-only pilot without offsets or flow.

An independent nearest-mean ablation and calibration-neighbor transfer isolate
order constraints from local denoising and extra query observations. Require
strong source-only and pooled scFates/DPT references plus segment/coverage oracles.
Report point and assignment-averaged predictions, paired reads, gene-fold rank
agreement and partial-calibration sensitivity. Reused panels, shared exposure and
only two controls limit identification; no depth enters selection or validation.

**Pre-fit support correction:** partial calibration cannot place uncovered parts.
Every new method uses the same modeling-PC gate: fifteenth calibration-neighbor
distance below the calibration's 95th-percentile fifteenth-other-neighbor radius.
Outside this frozen support, keep the source-scFates assignment unchanged. Report
support by segment/read state; this is a heuristic, not proven physical overlap.


In [ ]:
from pathlib import Path
import argparse,os,sys,hashlib,io,json
from importlib.metadata import version
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0,str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs
from pseudospace.count_representation import fit_count_representation,transform_count_representation
from pseudospace.trajectory_benchmark import fit_scfates_axis,fit_dpt_axis,project_neighbor_axis,fit_unoriented_scfates_axis
from pseudospace.repeated_structure import fit_atlas
from pseudospace.ordered_registration import local_curve_means,register_ordered_means,calibration_support
from pseudospace.within_segment_validation import fit_segment_covariate_oracle,predict_segment_covariate_oracle
from pseudospace.stage_cache import cached_payload
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root',default=os.environ.get('PSEUDOSPACE_DATA_ROOT',str(REPO/'data')))
parser.add_argument('--results-root',default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT',str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'repeated_structure_partial_registration';OUT.mkdir(parents=True,exist_ok=True)
NOTEBOOK_LOGIC_VERSION='29.partial_registration.1'
CONTROLS=('Ctrl1A2','Ctrl1A4');SEED,N_COMPONENTS,N_FOLDS=15,15,3
GRID=np.linspace(0,1,101)
QUERY_GRID=np.linspace(0,1,31)
QUERY_BANDWIDTH=.08
MIN_EFFECTIVE_SUPPORT=12
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
VERSIONS={name:version(name) for name in ['scanpy','scFates','anndata','scikit-learn','elpigraph-python']}
CODE_FILES=[REPO/'pseudospace'/name for name in ['repeated_inputs.py','count_representation.py','trajectory_benchmark.py','repeated_structure.py','within_segment_validation.py','ordered_registration.py','stage_cache.py']]
CODE_DIGEST=hashlib.sha256(b''.join(path.read_bytes() for path in CODE_FILES)).hexdigest()
REFERENCE_OUT=RESULTS_ROOT/'repeated_structure_anatomy_initialization'
REFERENCE_FILES={name:REFERENCE_OUT/(name+'.csv') for name in ['scores','thin_scores','selected_scores','thin_selected_scores','positions']}
if any(not p.is_file() for p in REFERENCE_FILES.values()):
    raise FileNotFoundError('Completed notebook-28 references required')
reference={name:pd.read_csv(path) for name,path in REFERENCE_FILES.items()}
REFERENCE_MANIFEST=REFERENCE_OUT/'input_manifest.json'
reference_manifest=json.loads(REFERENCE_MANIFEST.read_text())
if reference_manifest['config']['versions']!=VERSIONS:
    raise ValueError('Notebook-28 package versions changed; pooled context cannot be reused')
PANEL_PATH=RESULTS_ROOT/'within_segment_repeated_structure_validation'/'selected_genes.csv'
frozen_panel=pd.read_csv(PANEL_PATH,usecols=['source_specimen','test_specimen','fold','segment','gene','training_cv_gain'])
if frozen_panel.duplicated(['source_specimen','test_specimen','fold','segment','gene']).any():
    raise ValueError('Duplicated frozen program gene')
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,controls=CONTROLS,seed=SEED,components=N_COMPONENTS,
 folds=N_FOLDS,markers=sorted(MARKERS),count_theta=100.,graph=dict(nodes=30,elastic_lambda=.01,elastic_mu=.1,dpt_neighbors=30),
 query_curve='unlabeled_arbitrary_tip',query_knots=QUERY_GRID.tolist(),query_bandwidth=QUERY_BANDWIDTH,
 minimum_effective_support=MIN_EFFECTIVE_SUPPORT,registration='free_endpoints_monotone_both_orientations',
 registration_cost='mean_squared_PC_distance',registration_transition_penalty=0.,
 calibration_hash='sha256(structure_id) first8 %2==0',evaluation_hash='complement',regimes=['full','S3_only'],
 query_anatomy='not_passed',projection_neighbors=15,support_quantile=.95,support_fallback='source_scFates_assignments',
 reference_manifest_digest=hashlib.sha256(REFERENCE_MANIFEST.read_bytes()).hexdigest(),response_ridge=.001,response_grid=GRID.tolist(),
 response_reduction=['point','assignment_average'],quarter_read_fraction=.25,quarter_read_seed_rule='seed+fold',
 oracle=dict(degree=2,ridge=.01),panel_digest=hashlib.sha256(PANEL_PATH.read_bytes()).hexdigest(),versions=VERSIONS,
 reference_digests={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in REFERENCE_FILES.items()},depth_used=False)
print('Installed versions:',VERSIONS)


## Actual healthy cross-sections and fixed input guards

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if set(np.unique(specimens))!=set(CONTROLS): raise ValueError('Expected exactly two healthy controls')
if not np.isin(labels,['S1','S2','S3']).all(): raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[x] for x in labels],dtype=int)
genes=healthy.var_names.to_numpy(str)
X=healthy.layers['lognorm']
X=X.toarray().astype(np.float64,copy=False) if sparse.issparse(X) else np.asarray(X,dtype=np.float64)
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()
if 'n_spots' not in obs: raise ValueError('Required n_spots metadata is missing')
n_spots=pd.to_numeric(obs.n_spots,errors='coerce').to_numpy(float)
if not np.isfinite(n_spots).all() or np.any(n_spots<0): raise ValueError('Invalid n_spots metadata')
if not np.isfinite(X).all() or not np.isfinite(raw_total).all() or np.any(raw_total<=0):
    raise ValueError('Invalid expression or all-gene exposure')
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,
                code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),depth_used=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab(specimens,labels))

# Reused pooled context must describe the exact same prepared private inputs.
current_core=json.loads(json.dumps(healthy.uns['repeated_mouse_input_manifest'],default=str))
previous_core={}
for field in current_core:
    previous_core[field]=reference_manifest[field]
if current_core!=previous_core:
    raise ValueError('Notebook-28 input manifest changed; pooled context cannot be reused')


In [ ]:
def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def fit_gene_spline(response,z,sample):
    return fit_atlas(response,z,sample,grid=GRID,ridge=.001)

def normalized_variance(response,sample):
    mean=np.mean([response[sample==s].mean(axis=0) for s in np.unique(sample)],axis=0)
    return np.maximum(np.mean([((response[sample==s]-mean)**2).mean(axis=0)
                               for s in np.unique(sample)],axis=0),1e-3)

def primary_rows(response,predictions,variance,query_anatomy,meta):
    out=[]
    for method,prediction in predictions.items():
        scaled=(response-prediction)**2/variance
        for seg,name in [(None,'all'),(0,'S1'),(1,'S2'),(2,'S3')]:
            use=np.ones(len(response),bool) if seg is None else query_anatomy==seg
            if use.any():
                out.append(dict(**meta,method=method,segment=name,n=int(use.sum()),
                                normalized_mse=float(scaled[use].mean())))
    return out

def selected_rows(response,predictions,train_response,train_anatomy,eval_anatomy,panel,gene_to_j,meta):
    out=[]
    for seg,name in enumerate(['S1','S2','S3']):
        selected=panel[panel.segment==name]
        if selected.empty or not set(selected.gene).issubset(gene_to_j):
            raise ValueError('Invalid frozen panel for '+name)
        jj=np.array([gene_to_j[g] for g in selected.gene],int)
        variance=np.maximum(np.var(train_response[train_anatomy==seg],axis=0),1e-3)[jj]
        use=eval_anatomy==seg
        if not use.any(): raise ValueError('Missing evaluation segment '+name)
        for method,prediction in predictions.items():
            out.append(dict(**meta,method=method,segment=name+'_selected',n=int(use.sum()),
              selected_gene_count=len(jj),normalized_mse=float(np.mean(
               (response[use][:,jj]-prediction[use][:,jj])**2/variance))))
    return out


def assignment_predictions(assignments,decoders):
    out={}
    for method,a in assignments.items():
        out[method]=decoders[method].predict(a['z_mean'])
        out[method+'_assignment']=decoders[method].predict_distribution(a['support'],a['weights'])
    return out


## Separate specimen paths and partial registration

In [ ]:
def run_partial_registration():
    names = ['scores', 'thin_scores', 'selected_scores', 'thin_selected_scores',
             'positions', 'stability', 'diagnostics', 'failures', 'registration_knots']
    frames = {name: [] for name in names}
    from pseudospace.ordered_registration import calibration_support

    def fail(key, stage, regime, method, exc):
        frames['failures'].append(dict(**key, stage=stage, regime=regime,
                                       method=method, error=str(exc)))

    def apply_source_fallback(assignment, source_assignment, supported):
        result = {name: np.asarray(value).copy() for name, value in assignment.items()}
        for name in ('z_mean', 'support', 'weights'):
            result[name][~supported] = source_assignment[name][~supported]
        return result

    for source in CONTROLS:
        target = next(name for name in CONTROLS if name != source)
        tr, te = specimens == source, specimens == target
        query_ids = ids[te]
        calibration = np.array([
            int(hashlib.sha256(value.encode()).hexdigest()[:8], 16) % 2 == 0
            for value in query_ids
        ])
        evaluation = ~calibration
        for fold in range(N_FOLDS):
            key = dict(source_specimen=source, test_specimen=target, fold=fold)
            try:
                mg = np.flatnonzero(~marker_mask & (gene_fold != fold))
                eg = np.flatnonzero(~marker_mask & (gene_fold == fold))
                C, Q = dense_panel(raw_counts, tr, mg), dense_panel(raw_counts, te, mg)
                Y, transform = fit_count_representation(
                    C, raw_total[tr], specimens[tr], N_COMPONENTS, theta=100., seed=SEED)
                qY = transform_count_representation(Q, raw_total[te], transform)
                sa = anatomy[tr]
                response, qresponse = X[tr][:, eg], X[te][:, eg]
                variance = normalized_variance(response, specimens[tr])
                panel = frozen_panel[(frozen_panel.source_specimen == source) &
                                     (frozen_panel.test_specimen == target) &
                                     (frozen_panel.fold == fold)]
                gene_to_j = {gene: j for j, gene in enumerate(genes[eg])}
                full = sparse.csr_matrix(raw_counts[te])
                rng = np.random.default_rng(SEED + fold)
                thin = sparse.csr_matrix((
                    rng.binomial(full.data.astype(np.int64), .25),
                    full.indices.copy(), full.indptr.copy()), shape=full.shape)
                thinlib = np.asarray(thin.sum(axis=1)).ravel().astype(float)
                if np.any(thinlib <= 0):
                    raise ValueError('Quarter-read thinning produced zero exposure')
                thinC = dense_panel(thin, np.ones(len(thinlib), dtype=bool), mg)
                thin_qY = transform_count_representation(thinC, thinlib, transform)
                evalY, thin_evalY = qY[evaluation], thin_qY[evaluation]
                eval_response = qresponse[evaluation]
                eval_a, eval_ids = anatomy[te][evaluation], query_ids[evaluation]
                detected_train = np.asarray((raw_counts[tr][:, mg] > 0).sum(axis=1)).ravel()
                detected_query = np.asarray((raw_counts[te][:, mg] > 0).sum(axis=1)).ravel()
                train_cov = np.column_stack([np.log(raw_total[tr]), np.log1p(n_spots[tr]),
                                             np.log1p(detected_train)])
                query_cov = np.column_stack([np.log(raw_total[te]), np.log1p(n_spots[te]),
                                             np.log1p(detected_query)])
                coverage_model = fit_segment_covariate_oracle(
                    response, train_cov, sa, specimens[tr], ridge=.01, degree=2)
                coverage_all = predict_segment_covariate_oracle(
                    coverage_model, query_cov, anatomy[te])
                segmeans = {label: response[sa == label].mean(axis=0) for label in (0, 1, 2)}
                segment_all = np.vstack([segmeans[int(label)] for label in anatomy[te]])
            except Exception as exc:
                fail(key, 'sourcefold_setup', '', 'sourcefold_setup', exc)
                frames['diagnostics'].append(dict(**key, stage='source', regime='',
                    method='sourcefold_setup', status='failed', metadata='{}', error=str(exc)))
                print('Failed:', source, 'to', target, 'fold', fold, str(exc), flush=True)
                continue

            # Fit source-only axes and decoders once. A failed source arm does not
            # leak partial projections or stop the other reference method.
            source_axes, source_decoders = {}, {}
            for method, algorithm in [('source_scFates_neighbor', 'scFates'),
                                      ('source_DPT_neighbor', 'DPT')]:
                try:
                    if algorithm == 'scFates':
                        z, info = fit_scfates_axis(Y, sa, nodes=30, seed=SEED)
                    else:
                        z, info = fit_dpt_axis(Y, sa, n_neighbors=30, seed=SEED)
                    decoder = fit_gene_spline(response, z, specimens[tr])
                    source_axes[method] = z
                    source_decoders[method] = decoder
                    frames['diagnostics'].append(dict(**key, stage='source', regime='',
                        method=method, status='complete', metadata=json.dumps(info, sort_keys=True)))
                except Exception as exc:
                    fail(key, 'source', '', method, exc)
                    frames['diagnostics'].append(dict(**key, stage='source', regime='',
                        method=method, status='failed', metadata='{}', error=str(exc)))
                    print('Failed:', source, 'to', target, 'fold', fold, method, str(exc), flush=True)

            # PC mean reference for registration, fit only to the source scFates axis.
            pc_decoder = None
            source_pc_means = None
            source_pc_residual_mse = np.nan
            if 'source_scFates_neighbor' in source_axes:
                try:
                    pc_decoder = fit_gene_spline(
                        Y, source_axes['source_scFates_neighbor'], specimens[tr])
                    source_pc_means = pc_decoder.predict(GRID)
                    source_pc_residual_mse = float(np.mean((
                        Y - pc_decoder.predict(source_axes['source_scFates_neighbor'])) ** 2))
                except Exception as exc:
                    fail(key, 'source', '', 'source_pc_mean_decoder', exc)
                    pc_decoder = None
            else:
                source_pc_means = None
                source_pc_residual_mse = np.nan

            for regime in ('full', 'S3_only'):
                meta = dict(**key, regime=regime)
                cal = calibration if regime == 'full' else calibration & (anatomy[te] == 2)
                qcal = qY[cal]
                method_assignments, thin_assignments = {}, {}
                method_decoders = {}

                # Fixed source-only neighbors are scored even if all registration
                # operations below fail.
                for method, z in source_axes.items():
                    try:
                        assignment = project_neighbor_axis(
                            Y, z, evalY, k=15, return_distribution=True)
                        thin_assignment = project_neighbor_axis(
                            Y, z, thin_evalY, k=15, return_distribution=True)
                        method_assignments[method] = assignment
                        thin_assignments[method] = thin_assignment
                        method_decoders[method] = source_decoders[method]
                    except Exception as exc:
                        fail(key, 'source_projection', regime, method, exc)

                query_diag = dict(**key, stage='query', regime=regime,
                    method='query_curve_registration', qcal_n=int(cal.sum()),
                    status='failed', metadata='{}', qcal_radius95=np.nan,
                    calibration_support_fraction=np.nan, thinned_calibration_support_fraction=np.nan,
                    min_effective_support=np.nan,
                    ordered_cost=np.nan, independent_cost=np.nan, orientation=np.nan,
                    orientation_gap=np.nan, reference_span=np.nan,
                    distinct_reference_knots=np.nan, plateau_fraction=np.nan,
                    max_reference_jump=np.nan, source_pc_residual_mse=source_pc_residual_mse,
                    qcal_registered_pc_residual_mse=np.nan)
                if not cal.any():
                    exc = ValueError('empty calibration population')
                    query_diag['error'] = str(exc)
                    frames['diagnostics'].append(query_diag)
                    fail(key, 'query', regime, 'calibration_population', exc)
                else:
                    # Support is defined from calibration modeling PCs only; the
                    # helper self-excludes each calibration row for its radius.
                    try:
                        support_info = calibration_support(
                            qcal, np.vstack([evalY, thin_evalY]), k=15, quantile=.95)
                        n_eval = len(evalY)
                        calibration_supported = support_info['supported'][:n_eval]
                        thinned_calibration_supported = support_info['supported'][n_eval:]
                        radius95 = float(support_info['radius'])
                        query_diag.update(
                            qcal_radius95=radius95,
                            calibration_support_fraction=float(calibration_supported.mean()),
                            thinned_calibration_support_fraction=float(thinned_calibration_supported.mean()))
                    except Exception as exc:
                        calibration_supported = thinned_calibration_supported = None
                        query_diag['error'] = 'calibration support gate failed: ' + str(exc)
                        fail(key, 'query_support_gate', regime, 'calibration_support_gate', exc)

                    # Neighbor transfer first assigns each calibration profile to
                    # the frozen source axis; evaluation is extended from qcal only.
                    if calibration_supported is not None and 'source_scFates_neighbor' in source_axes:
                        try:
                            calibration_z = project_neighbor_axis(
                                Y, source_axes['source_scFates_neighbor'], qcal, k=15)
                            assignment = project_neighbor_axis(
                                qcal, calibration_z, evalY, k=15, return_distribution=True)
                            thin_assignment = project_neighbor_axis(
                                qcal, calibration_z, thin_evalY, k=15, return_distribution=True)
                            source_eval = method_assignments['source_scFates_neighbor']
                            source_thin = thin_assignments['source_scFates_neighbor']
                            assignment = apply_source_fallback(assignment, source_eval,
                                                               calibration_supported)
                            thin_assignment = apply_source_fallback(
                                thin_assignment, source_thin, thinned_calibration_supported)
                            method_assignments['calibration_neighbor_transfer'] = assignment
                            thin_assignments['calibration_neighbor_transfer'] = thin_assignment
                            method_decoders['calibration_neighbor_transfer'] = \
                                source_decoders['source_scFates_neighbor']
                        except Exception as exc:
                            fail(key, 'query_projection', regime,
                                 'calibration_neighbor_transfer', exc)

                    # The query curve receives expression-only calibration rows.
                    try:
                        if calibration_supported is None:
                            raise ValueError('calibration support gate unavailable')
                        if pc_decoder is None:
                            raise ValueError('source PC mean decoder unavailable')
                        query_native_z, query_meta = fit_unoriented_scfates_axis(
                            qcal, nodes=30, seed=SEED)
                        query_means, effective = local_curve_means(
                            qcal, query_native_z, QUERY_GRID, bandwidth=QUERY_BANDWIDTH)
                        query_diag['min_effective_support'] = float(np.min(effective))
                        query_diag['metadata'] = json.dumps(query_meta, sort_keys=True)
                        if np.any(effective < MIN_EFFECTIVE_SUPPORT):
                            query_diag['status'] = 'unsupported_local_support'
                            query_diag['error'] = (
                                f'minimum effective support {effective.min():.6g} below '
                                f'{MIN_EFFECTIVE_SUPPORT}')
                            fail(key, 'query', regime,
                                 'query_curve_registration', query_diag['error'])
                        else:
                            registration = register_ordered_means(
                                query_means, source_pc_means, GRID)
                            query_diag.update(
                                status='complete', ordered_cost=registration['cost'],
                                independent_cost=registration['independent_cost'],
                                orientation=registration['orientation'],
                                orientation_gap=registration['orientation_gap'],
                                reference_span=registration['reference_span'],
                                distinct_reference_knots=registration['distinct_reference_knots'],
                                plateau_fraction=registration['plateau_fraction'],
                                max_reference_jump=registration['max_reference_jump'],
                                qcal_registered_pc_residual_mse=float(np.mean((
                                    qcal - pc_decoder.predict(np.interp(query_native_z, QUERY_GRID, registration['z']))) ** 2)))
                            for j, native in enumerate(QUERY_GRID):
                                frames['registration_knots'].append(dict(
                                    **meta, query_native_z=float(native),
                                    mapped_z=float(registration['z'][j]),
                                    independent_z=float(registration['independent_z'][j]),
                                    effective_n=float(effective[j])))
                            # Map arbitrary native query coordinates through each
                            # saved correspondence; evaluation never refits it.
                            coordinate_methods = {
                                'ordered_partial_curve': registration['z'],
                                'independent_curve_mean_match': registration['independent_z'],
                            }
                            for method, knot_coords in coordinate_methods.items():
                                try:
                                    cal_z = np.interp(query_native_z, QUERY_GRID, knot_coords)
                                    assignment = project_neighbor_axis(
                                        qcal, cal_z, evalY, k=15, return_distribution=True)
                                    thin_assignment = project_neighbor_axis(
                                        qcal, cal_z, thin_evalY, k=15, return_distribution=True)
                                    source_eval = method_assignments['source_scFates_neighbor']
                                    source_thin = thin_assignments['source_scFates_neighbor']
                                    assignment = apply_source_fallback(
                                        assignment, source_eval, calibration_supported)
                                    thin_assignment = apply_source_fallback(
                                        thin_assignment, source_thin, thinned_calibration_supported)
                                    method_assignments[method] = assignment
                                    thin_assignments[method] = thin_assignment
                                    method_decoders[method] = source_decoders['source_scFates_neighbor']
                                except Exception as exc:
                                    fail(key, 'query_projection', regime, method, exc)
                    except Exception as exc:
                        query_diag['status'] = 'failed'
                        query_diag['error'] = str(exc)
                        fail(key, 'query_curve_fit', regime,
                             'query_curve_registration', exc)
                    frames['diagnostics'].append(query_diag)

                # Response targets remain original query measurements after thinning.
                predictions = assignment_predictions(method_assignments, method_decoders)
                thinned_predictions = assignment_predictions(thin_assignments, method_decoders)
                predictions['segment_coverage_oracle'] = coverage_all[evaluation]
                predictions['segment_mean_oracle'] = segment_all[evaluation]
                thinned_predictions['segment_coverage_oracle'] = coverage_all[evaluation]
                thinned_predictions['segment_mean_oracle'] = segment_all[evaluation]
                frames['scores'].extend(primary_rows(
                    eval_response, predictions, variance, eval_a, meta))
                frames['thin_scores'].extend(primary_rows(
                    eval_response, thinned_predictions, variance, eval_a, meta))
                frames['selected_scores'].extend(selected_rows(
                    eval_response, predictions, response, sa, eval_a, panel, gene_to_j, meta))
                frames['thin_selected_scores'].extend(selected_rows(
                    eval_response, thinned_predictions, response, sa, eval_a, panel, gene_to_j, meta))
                for method, assignment in method_assignments.items():
                    zq = assignment['z_mean']
                    zthin = thin_assignments[method]['z_mean']
                    for segment, label in [(None, 'all'), (0, 'S1'), (1, 'S2'), (2, 'S3')]:
                        use = np.ones(len(zq), dtype=bool) if segment is None else eval_a == segment
                        if use.any():
                            frames['stability'].append(dict(
                                **meta, method=method, segment=label, n=int(use.sum()),
                                coordinate_mae=float(np.mean(np.abs(zq[use] - zthin[use]))),
                                calibration_supported_fraction=float(
                                    calibration_supported[use].mean()) if method in (
                                        'calibration_neighbor_transfer', 'ordered_partial_curve',
                                        'independent_curve_mean_match') and calibration_supported is not None else 1.,
                                thinned_calibration_supported_fraction=float(
                                    thinned_calibration_supported[use].mean()) if method in (
                                        'calibration_neighbor_transfer', 'ordered_partial_curve',
                                        'independent_curve_mean_match') and thinned_calibration_supported is not None else 1.))
                    is_new_method = method in ('calibration_neighbor_transfer',
                        'ordered_partial_curve', 'independent_curve_mean_match')
                    for row_i, (sid, value, thin_value, lab) in enumerate(zip(eval_ids, zq, zthin, eval_a)):
                        frames['positions'].append(dict(
                            **meta, method=method, structure_id=sid, z=float(value),
                            thinned_z=float(thin_value), segment=['S1', 'S2', 'S3'][int(lab)],
                            calibration_supported=bool(calibration_supported[row_i]) if is_new_method and calibration_supported is not None else True,
                            thinned_calibration_supported=bool(thinned_calibration_supported[row_i]) if is_new_method and thinned_calibration_supported is not None else True))
                print('Completed:', source, 'to', target, 'fold', fold, regime, flush=True)
    return {name: pd.DataFrame(rows).to_json(orient='table')
            for name, rows in frames.items()}


In [ ]:
payload=cached_payload('partial_registration',run_partial_registration,
 root=OUT/'stage_cache',params=CACHE_PARAMS,
 inputs=dict(X=X,counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,anatomy=anatomy,
  markers=marker_mask,gene_fold=gene_fold,n_spots=n_spots,panel=frozen_panel,reference=reference),code=CODE_DIGEST)
outputs={name:pd.read_json(io.StringIO(str(value)),orient='table') for name,value in payload.items()}
for name,frame in outputs.items():frame.to_csv(OUT/(name+'.csv'),index=False)
display(outputs['diagnostics']);display(outputs['failures'])
score_keys=['source_specimen','test_specimen','fold','regime','method','segment']
position_keys=['source_specimen','test_specimen','fold','regime','method','structure_id']
base=['source_scFates_neighbor','source_DPT_neighbor']
score_base=base+[m+'_assignment' for m in base]+['segment_mean_oracle','segment_coverage_oracle']
# Uncached guards also validate cached data before the older pooled context is used.
for name in ['scores','thin_scores','selected_scores','thin_selected_scores','positions']:
    methods=base if name=='positions' else score_base
    actual=outputs[name][outputs[name].method.isin(methods)]
    previous=reference[name][reference[name].method.isin(methods)]
    keys=position_keys if name=='positions' else score_keys
    columns=['z','thinned_z'] if name=='positions' else ['normalized_mse']
    matched=actual.merge(previous,on=keys,validate='one_to_one',suffixes=('_new','_old'))
    if len(matched)!=len(actual) or len(matched)!=len(previous):
        raise ValueError('Source/oracle reference row/ID mismatch: '+name)
    for column in columns:
        if not np.allclose(matched[column+'_new'],matched[column+'_old'],rtol=1e-8,atol=1e-10):
            raise ValueError('Uncached source/oracle reproduction failed: '+name+' '+column)
# Unsupported calibration projections retain the exact source position, also after cache hits.
source_positions=outputs['positions'][outputs['positions'].method=='source_scFates_neighbor']
for method in ['calibration_neighbor_transfer','ordered_partial_curve','independent_curve_mean_match']:
    actual=outputs['positions'][outputs['positions'].method==method]
    matched=actual.merge(source_positions,on=[k for k in position_keys if k!='method'],
                         validate='one_to_one',suffixes=('_new','_source'))
    if len(matched)!=len(actual):raise ValueError('Missing fallback source IDs')
    for column,flag in [('z','calibration_supported'),('thinned_z','thinned_calibration_supported')]:
        unsupported=~matched[flag+'_new'].astype(bool)
        if not np.array_equal(matched.loc[unsupported,column+'_new'],matched.loc[unsupported,column+'_source']):
            raise ValueError('Uncached unsupported-source fallback failed: '+method+' '+column)
diag=outputs['diagnostics']
source_diag=diag[diag.stage=='source']
if len(source_diag)!=12 or not (source_diag.status=='complete').all():
    raise RuntimeError('Incomplete source references; inspect failures')
query_diag=diag[diag.stage=='query']
if len(query_diag)!=12:raise RuntimeError('Missing query-curve attempt records')
print('Completed: uncached source/oracle scores, program, ID, coordinate and read reproduction guards')
display(diag.groupby(['stage','method','status']).size().rename('attempt_count').reset_index())
for key,group in outputs['registration_knots'].reindex(columns=['source_specimen','test_specimen','fold','regime','query_native_z','mapped_z','independent_z','effective_n']).groupby(['source_specimen','test_specimen','fold','regime']):
    record=query_diag.set_index(['source_specimen','test_specimen','fold','regime']).loc[key]
    if len(group)!=len(QUERY_GRID) or np.any(np.diff(group.mapped_z)*record.orientation < -1e-12):
        raise ValueError('Uncached ordered-registration knot guard failed')
display(outputs['stability'].groupby(['source_specimen','regime','method','segment']).agg(
 coordinate_mae=('coordinate_mae','mean'),
 support_fraction=('calibration_supported_fraction','mean'),
 thinned_support_fraction=('thinned_calibration_supported_fraction','mean')))
# Reuse expensive pooled baselines only after exact input-manifest and source guards.
pooled=['pooled_scFates_neighbor','pooled_DPT_neighbor']
pooled_scores=pooled+[m+'_assignment' for m in pooled]
combined={}
for name in ['scores','thin_scores','selected_scores','thin_selected_scores','positions']:
    methods=pooled if name=='positions' else pooled_scores
    combined[name]=pd.concat([outputs[name],reference[name][reference[name].method.isin(methods)]],ignore_index=True)
    combined[name].to_csv(OUT/(name+'_with_context.csv'),index=False)
keys=['source_specimen','test_specimen','fold','regime','segment']
pairs=[]
for suffix in ['', '_assignment']:
    for candidate,baselines in [
       ('ordered_partial_curve',['independent_curve_mean_match','calibration_neighbor_transfer','source_scFates_neighbor','source_DPT_neighbor','pooled_scFates_neighbor','pooled_DPT_neighbor']),
       ('independent_curve_mean_match',['calibration_neighbor_transfer','source_scFates_neighbor']),
       ('calibration_neighbor_transfer',['source_scFates_neighbor'])]:
        pairs.extend([(candidate+suffix,baseline+suffix) for baseline in baselines])
    pairs.extend([('ordered_partial_curve'+suffix,b) for b in ['segment_mean_oracle','segment_coverage_oracle']])
for name in ['scores','thin_scores','selected_scores','thin_selected_scores']:
    frame=combined[name];parts=[]
    for candidate,baseline in pairs:
        left=frame[frame.method==candidate][keys+['normalized_mse']]
        right=frame[frame.method==baseline][keys+['normalized_mse']]
        match=left.merge(right,on=keys,validate='one_to_one',suffixes=('_candidate','_baseline'))
        match['candidate']=candidate;match['baseline']=baseline
        match['gain_percent']=100*(1-match.normalized_mse_candidate/match.normalized_mse_baseline)
        parts.append(match)
    matched=pd.concat(parts,ignore_index=True);matched.to_csv(OUT/(name+'_matched.csv'),index=False)
    summary=matched.groupby(['source_specimen','regime','segment','candidate','baseline']).agg(
        gain_percent=('gain_percent','mean'),fold_count=('fold','nunique')).reset_index()
    summary.to_csv(OUT/(name+'_summary.csv'),index=False);display(summary)
positions=combined['positions'];rank_rows=[]
for key,frame in positions.groupby(['source_specimen','test_specimen','regime','method','segment']):
    wide=frame.pivot(index='structure_id',columns='fold',values='z')
    for i in range(N_FOLDS):
        for j in range(i+1,N_FOLDS):
            if i not in wide or j not in wide:continue
            paired=wide[[i,j]].dropna()
            rank_rows.append(dict(zip(['source_specimen','test_specimen','regime','method','segment'],key),
              fold_a=i,fold_b=j,n=len(paired),spearman=float(spearmanr(paired[i],paired[j]).statistic)))
rank_agreement=pd.DataFrame(rank_rows);rank_agreement.to_csv(OUT/'gene_fold_rank_agreement.csv',index=False)
if not rank_agreement.empty:
    display(rank_agreement.groupby(['source_specimen','regime','method','segment']).agg(
      spearman=('spearman','mean'),valid_pair_count=('spearman','count'),pair_count=('fold_a','size')))
sensitivity=positions[positions.regime=='full'].merge(
 positions[positions.regime=='S3_only'],on=[k for k in position_keys if k!='regime'],validate='one_to_one',suffixes=('_full','_partial'))
if sensitivity.empty:raise ValueError('Missing full/partial calibration comparison IDs')
sensitivity['coordinate_change']=abs(sensitivity.z_full-sensitivity.z_partial)
sensitivity=sensitivity.groupby(['source_specimen','method']).agg(
 coordinate_mae=('coordinate_change','mean'),n=('structure_id','size'),fold_count=('fold','nunique')).reset_index()
sensitivity.to_csv(OUT/'calibration_sensitivity.csv',index=False);display(sensitivity)
import matplotlib.pyplot as plt
program=pd.read_csv(OUT/'selected_scores_matched.csv')
fig,axes=plt.subplots(2,2,figsize=(11,7),sharey=False)
plot_methods=[('ordered_partial_curve_assignment','Ordered registration'),
 ('independent_curve_mean_match_assignment','Independent means'),
 ('calibration_neighbor_transfer_assignment','Calibration neighbors')]
for row,source in enumerate(CONTROLS):
    for col,regime in enumerate(['full','S3_only']):
        ax=axes[row,col]
        for offset,color,(candidate,label) in zip([-.2,0,.2],['#0072B2','#D55E00','#009E73'],plot_methods):
            sub=program[(program.source_specimen==source)&(program.regime==regime)&
                (program.candidate==candidate)&(program.baseline=='source_scFates_neighbor_assignment')]
            for seg,name in enumerate(['S1_selected','S2_selected','S3_selected']):
                vals=sub[sub.segment==name].gain_percent
                if len(vals):
                    ax.scatter(np.repeat(seg+offset,len(vals)),vals,s=18,alpha=.65,color=color,label=label if seg==0 else None)
                    ax.plot([seg+offset-.06,seg+offset+.06],[vals.mean()]*2,color='black',lw=1)
        ax.axhline(0,color='gray',lw=.8);ax.set_xticks([0,1,2],['S1','S2','S3'])
        ax.set_title(source+' → held-out control; '+regime)
        if col==0:ax.set_ylabel('Program error reduction (%)')
        if row==1:ax.set_xlabel('Within-segment development program')
axes[0,0].legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/'partial_registration_program_gains.png',dpi=160)
display(fig);plt.close(fig)
knots=outputs['registration_knots']
if not knots.empty:
    fig,axes=plt.subplots(2,2,figsize=(9,6),sharex=True,sharey=True)
    for row,source in enumerate(CONTROLS):
        for col,regime in enumerate(['full','S3_only']):
            ax=axes[row,col]
            subset=knots[(knots.source_specimen==source)&(knots.regime==regime)]
            for fold,group in subset.groupby('fold'):
                ax.plot(group.query_native_z,group.mapped_z,label='Fold '+str(fold))
            ax.set_title(source+' reference; '+regime);ax.set_xlim(0,1);ax.set_ylim(0,1)
            if row==1:ax.set_xlabel('Native query-curve coordinate')
            if col==0:ax.set_ylabel('Mapped reference coordinate')
    axes[0,0].legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/'partial_registration_maps.png',dpi=160)
    display(fig);plt.close(fig)


## Measured decision

Eleven of twelve registrations pass the fixed local-support gate. Ctrl1A2
source/fold 2/S3-only has effective support 9.596 <12 and remains rejected.
All source/oracle and unsupported-row fallback checks pass, also on a cache hit.

Ordered full-calibration S1/S2/S3 program gains versus assignment-aware source
scFates are −2.07%/−2.78%/−2.92% and −3.97%/−7.81%/−8.29%, with three
folds per cell. The order constraint is inactive in all first-direction full
fits. Partial gains are 0%/−5.27%/−7.32% (two folds) and
−0.77%/−71.62%/−9.67% (three). Reverse partial S3 gene-fold rank agreement
falls to 0.049, versus source scFates 0.794. Partial maps occupy subranges rather
than the full reference, but repeated knots and fine-order failures remain.

The common support gate retains the source assignment outside molecular
calibration support. It nevertheless admits 21.2% of reverse S2 profiles under
S3-only calibration, increasing to 55.2% after thinning. This is an empirical
molecular-support heuristic, not physical overlap probability. Reverse partial
S2 program error worsens by 251% under quarter reads. Smaller calibration-regime
coordinate changes than pooled scFates do not imply better reconstruction.

**Do not adopt or expand this branch.** An independently fitted query curve is
not necessarily spatial within a truncated segment. Future inference should
retain useful local reference information and explicitly condition on the
observation process. The contribution of exposure-dependent representation to
false support remains untested; do not turn that hypothesis into a conclusion.

The two reused controls, source-selected development programs, source annotations
and shared all-gene exposure limit physical identification. Neighbor assignments
are not calibrated anatomical posteriors; gene-fold agreement is stability and
curve coordinate is not physical length. No depth, injury/species or regulator
claim follows. The repeated-structure reconstruction goal remains unmet.
